In [2]:
import pandas as pd
import re

# Input file name
input_file = "/home/tom/Downloads/testcd.csv"

# Read CSV (semicolon-separated, with quoted headers)
df = pd.read_csv(input_file, sep=';', quotechar='"')

# Ensure the Dataset column name is correct
# If needed, print(df.columns) to confirm the name
dataset_col = 'Dataset'

# Helper: extract tissue type from Dataset string
def get_tissue(dataset):
    if "Lunge" in dataset:
        return "Lung"
    if "Ohr" in dataset:
        return "Ear"
    if "Haut" in dataset:
        return "Skin"
    return "Other"

# Helper: extract numeric dose in Gy from substrings like '0Gy', '01Gy', '1Gy', '2Gy', '03Gy', etc.
dose_pattern = re.compile(r'(\d+\.?\d*)Gy')

def get_dose_gy(dataset):
    m = dose_pattern.search(dataset)
    if m:
        try:
            return float(m.group(1))
        except ValueError:
            return None
    return None

# Add helper columns
df['Tissue'] = df[dataset_col].apply(get_tissue)
df['Dose_Gy'] = df[dataset_col].apply(get_dose_gy)

# Filter only entries with a recognized tissue
valid = df['Tissue'].isin(['Lung', 'Skin', 'Ear'])
df = df[valid].copy()

# Optionally restrict to 0–2 Gy if that's what you want
# Comment this block out if you want to keep all doses.
dose_mask = (df['Dose_Gy'] >= 0.0) & (df['Dose_Gy'] <= 2.0)
df_0_2Gy = df[dose_mask].copy()

# Split per tissue, sort by dose
for tissue, out_name in [
    ('Lung', 'Lung_0-2Gy.csv'),
    ('Skin', 'Skin_0-2Gy.csv'),
    ('Ear',  'Ear_0-2Gy.csv')
]:
    sub = df_0_2Gy[df_0_2Gy['Tissue'] == tissue].copy()
    if sub.empty:
        continue
    sub = sub.sort_values(by='Dose_Gy', ascending=True)
    # Drop helper columns if you don't want them in the output
    sub_out = sub.drop(columns=['Tissue', 'Dose_Gy'])
    sub_out.to_csv(out_name, sep=';', index=False)

print("Done. Written Lung_0-2Gy.csv, Skin_0-2Gy.csv, Ear_0-2Gy.csv (if corresponding data existed).")


Done. Written Lung_0-2Gy.csv, Skin_0-2Gy.csv, Ear_0-2Gy.csv (if corresponding data existed).


In [25]:
import os
import re
import pandas as pd

# Directory containing your CSV files
INPUT_DIR = "/home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/"  # change if needed
os.chdir(INPUT_DIR)

OUT_LUNG = "All_Lung.csv"
OUT_SKIN = "All_Skin.csv"
OUT_EAR  = "All_Ear.csv"

# Helper: extract tissue type from Dataset string
def get_tissue(dataset):
    if isinstance(dataset, str):
        if "Lunge" in dataset:
            return "Lung"
        if "Ohr" in dataset:
            return "Ear"
        if "Haut" in dataset:
            return "Skin"
    return None

# Helper: extract numeric dose in Gy from substrings like '0Gy', '01Gy', '1Gy', '2Gy', '03Gy'
dose_pattern = re.compile(r'(\d+\.?\d*)Gy')

def get_dose_gy_str(dataset):
    if not isinstance(dataset, str):
        return None
    m = dose_pattern.search(dataset)
    if m:
        return m.group(1)  # keep as string, e.g. '0', '01', '2', '03'
    return None

lung_frames = []
skin_frames = []
ear_frames  = []

for fname in os.listdir(INPUT_DIR):
    if not fname.lower().endswith(".csv"):
        continue

    fpath = os.path.join(INPUT_DIR, fname)
    print(f"Processing: {fpath}")

    try:
        # Read everything as string to preserve original formatting
        df = pd.read_csv(fpath, sep=';', quotechar='"', dtype=str)
    except Exception as e:
        print(f"  Skipping {fname}, read error: {e}")
        continue

    if "Dataset" not in df.columns:
        print(f"  Skipping {fname}, no 'Dataset' column.")
        continue

    df["Tissue"] = df["Dataset"].apply(get_tissue)
    df["Dose_Gy_str"] = df["Dataset"].apply(get_dose_gy_str)
    df = df[df["Tissue"].notna()].copy()

    lung_frames.append(df[df["Tissue"] == "Lung"])
    skin_frames.append(df[df["Tissue"] == "Skin"])
    ear_frames.append(df[df["Tissue"] == "Ear"])

def german_int_cleanup(s):
    """
    Take a string that may contain dots as thousands separators (e.g. '18.865')
    and return a pure integer string '18865'. Leave non-numeric strings untouched.
    """
    if not isinstance(s, str):
        return s
    # Remove surrounding quotes if any
    s = s.strip()
    # Remove dots used as thousand separators
    s_clean = s.replace(".", "")
    # If result is all digits, keep it
    if s_clean.isdigit():
        return s_clean
    return s  # leave untouched if not pure digits

def concat_and_save(frames, out_name):
    if not frames:
        print(f"No data for {out_name}, file not created.")
        return

    big = pd.concat(frames, ignore_index=True)

    # Sort by dose if available (convert Dose_Gy_str to float only for sorting)
    if "Dose_Gy_str" in big.columns:
        def dose_to_float(d):
            if d is None or not isinstance(d, str):
                return float("inf")
            try:
                return float(d.replace(",", "."))
            except ValueError:
                return float("inf")
        big["_dose_sort"] = big["Dose_Gy_str"].apply(dose_to_float)
        big = big.sort_values(by="_dose_sort", ascending=True)
        big = big.drop(columns=["_dose_sort"])

    # Drop helper columns before output if not needed
    big = big.drop(columns=["Tissue", "Dose_Gy_str"], errors="ignore")

    # Clean all columns except 'Dataset': remove thousand-separator dots in numeric-looking strings
    for col in big.columns:
        if col == "Dataset":
            continue
        big[col] = big[col].map(german_int_cleanup)

    out_path = os.path.join(INPUT_DIR, out_name)
    big.to_csv(out_path, sep=';', index=False)
    print(f"Written {out_path} with {len(big)} rows.")

concat_and_save(lung_frames, OUT_LUNG)
concat_and_save(skin_frames, OUT_SKIN)
concat_and_save(ear_frames, OUT_EAR)

print("Done.")


Processing: /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/14-Haut_nonTNF_0Gy-A1-AxVPI&14-Haut_nonTNF_01Gy-A7-AxVPI&14-Haut_nonTNF_1Gy-C1-AxVPI&14-Haut_nonTNF_...-Information.csv
Processing: /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/18-Ohr_nonTNF_0Gy-A2-AxVPI&18-Ohr_nonTNF_01Gy-A8-AxVPI&18-Ohr_nonTNF_1Gy-C2-AxVPI&18-Ohr_nonTNF_2Gy-...-Information.csv
Processing: /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/15-Haut_nonTNF_0Gy-A1-AxVPI&15-Haut_nonTNF_01Gy-A7-AxVPI&15-Haut_nonTNF_1Gy-C1-AxVPI&15-Haut_nonTNF_...-Information.csv
Written /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/All_Lung.csv with 36 rows.
Written /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/All_Skin.csv with 36 rows.
Written /home/tom/Desktop/Experiments/Celldeath/Kaluzasf14w/24h/All_Ear.csv with 36 rows.
Done.
